## Hypotension Labels (Mimic)

In [ ]:
import pandas as pd
outcome_df = pd.read_csv('hypotension_si_labels_mimic.csv.gz')
outcome_df.columns = ['Time Stamp (seconds)', 'SBP (mmHg)', 'DBP (mmHg)', 'MBP (mmHg)', 'hr_bpm','hypotension','shock_index','shock_index_label',\
        'file_name', 'file_path', 'subject_id', 'date']
outcome_df['date'] = pd.to_datetime(outcome_df['date'])
outcome_df['Time Stamp (seconds)'] = outcome_df['Time Stamp (seconds)'].round()
outcome_df.sort_values(by=['subject_id', 'date', 'Time Stamp (seconds)'], inplace=True)
outcome_df['unique_id'] = outcome_df['subject_id'] + '-' + outcome_df['date'].astype(str)
# # require people ICU for at least 2 hours
ids_to_keep = outcome_df.loc[outcome_df['Time Stamp (seconds)'] >= 7200, 'unique_id'].unique().tolist()
outcome_df = outcome_df.loc[outcome_df['unique_id'].isin(ids_to_keep)].copy()

In [ ]:
import numpy as np
rolling_window = 5
outcome_df.loc[outcome_df['hypotension'] == 2, 'hypotension'] = np.nan # min_periods handles these nans, it ensures at least 5 minutes are present
# dont forget to shift it back!, without the shift 
outcome_df['rolling_5min_hypotension'] = outcome_df.groupby('unique_id')['hypotension'].transform(lambda x: x.rolling(rolling_window, step=1, center=False, min_periods=1).sum().shift(-(rolling_window-1)))

outcome_df['hypotension_5min_consecutive'] = (outcome_df['rolling_5min_hypotension'] == 5).astype(int) # five consecutive minutes of hypotension

In [ ]:
hypotensive_events = outcome_df.loc[outcome_df['hypotension_5min_consecutive'] == 1].copy()
hypotensive_events['timestamp_shift'] = hypotensive_events.groupby('unique_id')['Time Stamp (seconds)'].shift(1)
#hypotensive_events['timestamp_diff'] = hypotensive_events['timestamp_shift'] - hypotensive_events['Time Stamp (seconds)']
hypotensive_events['timestamp_diff'] =  hypotensive_events['Time Stamp (seconds)'] - hypotensive_events['timestamp_shift']
hypotensive_events['new_event'] = (hypotensive_events['timestamp_diff'] > 60).astype(int) # given the rolling window, its impossible to have a new event within 5 minutes, so anything greater than 60 is a new event (there are no events between 60 seconds and 5 minutes)
hypotensive_events['event_id'] = hypotensive_events.groupby('unique_id')['new_event'].cumsum() # create subject specific event ids
hypotensive_events['unique_event_id'] = hypotensive_events['unique_id'] + '-' + hypotensive_events['event_id'].astype(str) # create unique event ids
hypotensive_events['event_length'] = hypotensive_events.groupby('unique_event_id')['hypotension_5min_consecutive'].transform(lambda x: x.sum() * 60)

In [ ]:
hypotensive_events.sort_values(by=['unique_id', 'Time Stamp (seconds)'], inplace=True)
hypotensive_events.drop(columns=['timestamp_shift', 'timestamp_diff'], inplace=True)
hypotensive_events = hypotensive_events.groupby('unique_event_id', as_index=False).first() # subset to first event per unique_event_id (first event is when it starts)

In [ ]:
hypotensive_events['time_between_events'] = hypotensive_events.groupby('unique_id')['Time Stamp (seconds)'].diff()#.mean()

In [ ]:
mean_time_between_events = hypotensive_events.groupby('unique_id')['time_between_events'].mean().mean()
all_events_mean_time = hypotensive_events['Time Stamp (seconds)'].mean()
mean_hypotension_time = hypotensive_events.groupby('unique_id')['Time Stamp (seconds)'].mean().mean()
last_patient_hypotension_time = hypotensive_events.groupby('unique_id')['Time Stamp (seconds)'].last().mean()

print(all_events_mean_time / 3600, mean_hypotension_time / 3600, last_patient_hypotension_time / 3600, mean_time_between_events / 3600)

In [ ]:
# filter to people with no hypotension and filter out people who reached hypotension threshold
# this should be rolling_10min_hypotension < 5 bc hypotension_5min_in_10min_win converts NaNs to 0s from astype(int)
max_hypotension_minutes = 0 # number of acceptable minutes of hypotension (note that NaNs could be present in a 5min rolling window)
# but we will filter out the NaNs
non_hypotensive_events = outcome_df.loc[(outcome_df['rolling_5min_hypotension'] <= max_hypotension_minutes) &\
                                        (outcome_df['hypotension'].notna()) &\
                                         ~(outcome_df['unique_id'].isin(hypotensive_events['unique_id'].unique().tolist()))].copy()
non_hypotensive_events['timestamp_shift'] = non_hypotensive_events.groupby('unique_id')['Time Stamp (seconds)'].shift(1) # shift five minutes (to force new events, similar to how hypotensive events are defined)
non_hypotensive_events['timestamp_diff'] =  non_hypotensive_events['Time Stamp (seconds)'] - non_hypotensive_events['timestamp_shift']
non_hypotensive_events['new_event'] = (non_hypotensive_events['timestamp_diff'] >= 60).astype(int) # identify new events after 5 minutes
non_hypotensive_events['event_id'] = non_hypotensive_events.groupby('unique_id')['new_event'].cumsum() # create subject specific event ids
non_hypotensive_events['unique_event_id'] = non_hypotensive_events['unique_id'] + '-' + non_hypotensive_events['event_id'].astype(str) # create unique event ids
non_hypotensive_events['event_length'] = -1

In [ ]:
non_hypotensive_events.sort_values(by=['unique_id', 'Time Stamp (seconds)'], inplace=True)
non_hypotensive_events.drop(columns=['timestamp_shift', 'timestamp_diff'], inplace=True)
non_hypotensive_events = non_hypotensive_events.groupby('unique_event_id', as_index=False).first()

non_hypotensive_events.head()


In [ ]:
# filter to events that are before the average last hypotension time 
non_hypotensive_events = non_hypotensive_events.loc[non_hypotensive_events['Time Stamp (seconds)'] <= last_patient_hypotension_time]

In [ ]:
hypotensive_events['hypotension_label'] = 1
non_hypotensive_events['hypotension_label'] = 0
pd.concat([non_hypotensive_events, hypotensive_events]).to_csv('hypotension_labels_mimic_all_events_rolling5min.csv.gz', index=False, compression='gzip')

## Shock Index Labels (Mimic)

In [ ]:
# now same for shock index
import pandas as pd
outcome_df = pd.read_csv('hypotension_si_labels_mimic.csv.gz')
outcome_df.columns = ['Time Stamp (seconds)', 'SBP (mmHg)', 'DBP (mmHg)', 'MBP (mmHg)', 'hr_bpm','hypotension','shock_index','shock_index_label',\
        'file_name', 'file_path', 'subject_id', 'date']
outcome_df['date'] = pd.to_datetime(outcome_df['date'])
outcome_df['Time Stamp (seconds)'] = outcome_df['Time Stamp (seconds)'].round()
outcome_df.sort_values(by=['subject_id', 'date', 'Time Stamp (seconds)'], inplace=True)
outcome_df['unique_id'] = outcome_df['subject_id'] + '-' + outcome_df['date'].astype(str)
# require people ICU for at least 2 hours
ids_to_keep = outcome_df.loc[outcome_df['Time Stamp (seconds)'] >= 7200, 'unique_id'].unique().tolist()
outcome_df = outcome_df.loc[outcome_df['unique_id'].isin(ids_to_keep)].copy()
outcome_df = outcome_df.loc[outcome_df.hypotension.isin([0,1])].copy() # filter to valid bp 

In [ ]:
import numpy as np
rolling_window = 5
outcome_df['rolling_5min_shock'] = outcome_df.groupby('unique_id')['shock_index_label'].transform(lambda x: x.rolling(rolling_window, step=1, center=False, min_periods=1).sum().shift(-(rolling_window-1)))
outcome_df['shock_5min_consecutive'] = (outcome_df['rolling_5min_shock'] == 5).astype(int) # five consecutive minutes of hypotension

In [ ]:
shock_events = outcome_df.loc[outcome_df['shock_5min_consecutive'] == 1].copy()
shock_events['timestamp_shift'] = shock_events.groupby('unique_id')['Time Stamp (seconds)'].shift(1)
shock_events['timestamp_diff'] =  shock_events['Time Stamp (seconds)'] - shock_events['timestamp_shift']
shock_events['new_event'] = (shock_events['timestamp_diff'] > 60).astype(int) # given the rolling window, its impossible to have a new event within 5 minutes, so anything greater than 60 is a new event (there are no events between 60 seconds and 5 minutes)
shock_events['event_id'] = shock_events.groupby('unique_id')['new_event'].cumsum() # create subject specific event ids
shock_events['unique_event_id'] = shock_events['unique_id'] + '-' + shock_events['event_id'].astype(str) # create unique event ids
shock_events['event_length'] = shock_events.groupby('unique_event_id')['shock_5min_consecutive'].transform(lambda x: x.sum() * 60)

In [ ]:
shock_events.sort_values(by=['unique_id', 'Time Stamp (seconds)'], inplace=True)
shock_events.drop(columns=['timestamp_shift', 'hypotension', 'timestamp_diff'], inplace=True)
shock_events = shock_events.groupby('unique_event_id', as_index=False).first() # subset to first event per unique_event_id (first event is when it starts)

In [ ]:
shock_events['time_between_events'] = shock_events.groupby('unique_id')['Time Stamp (seconds)'].diff()#.mean()

In [ ]:
mean_time_between_events = shock_events.groupby('unique_id')['time_between_events'].mean().mean()
all_events_mean_time = shock_events['Time Stamp (seconds)'].mean()
mean_shock_time = shock_events.groupby('unique_id')['Time Stamp (seconds)'].mean().mean()
last_patient_shock_time = shock_events.groupby('unique_id')['Time Stamp (seconds)'].last().mean()

print(all_events_mean_time / 3600, mean_shock_time / 3600, last_patient_shock_time / 3600, mean_time_between_events / 3600)

In [ ]:
# filter to people with no hypotension and filter out people who reached hypotension threshold
# this should be rolling_10min_hypotension < 5 bc hypotension_5min_in_10min_win converts NaNs to 0s from astype(int)
max_shock_minutes = 0 # number of acceptable minutes of hypotension (note that NaNs could be present in a 5min rolling window)
# but we will filter out the NaNs
non_shock_events = outcome_df.loc[(outcome_df['rolling_5min_shock'] <= max_shock_minutes) &\
                                        (outcome_df['shock_index_label'].notna()) &\
                                         ~(outcome_df['unique_id'].isin(shock_events['unique_id'].unique().tolist()))].copy()
non_shock_events['timestamp_shift'] = non_shock_events.groupby('unique_id')['Time Stamp (seconds)'].shift(1) # shift five minutes (to force new events, similar to how hypotensive events are defined)
non_shock_events['timestamp_diff'] =  non_shock_events['Time Stamp (seconds)'] - non_shock_events['timestamp_shift']
non_shock_events['new_event'] = (non_shock_events['timestamp_diff'] >= 60).astype(int) # identify new events after 5 minutes
non_shock_events['event_id'] = non_shock_events.groupby('unique_id')['new_event'].cumsum() # create subject specific event ids
non_shock_events['unique_event_id'] = non_shock_events['unique_id'] + '-' + non_shock_events['event_id'].astype(str) # create unique event ids
non_shock_events['event_length'] = -1

In [ ]:
non_shock_events.sort_values(by=['unique_id', 'Time Stamp (seconds)'], inplace=True)
non_shock_events.drop(columns=['timestamp_shift', 'hypotension', 'timestamp_diff'], inplace=True)
non_shock_events = non_shock_events.groupby('unique_event_id', as_index=False).first()

non_shock_events.head()


In [ ]:
# filter to events that are before the average last hypotension time 
non_shock_events = non_shock_events.loc[non_shock_events['Time Stamp (seconds)'] <= last_patient_shock_time]

In [ ]:
shock_events['shock_index_label'] = 1
non_shock_events['shock_index_label'] = 0
pd.concat([non_shock_events, shock_events]).to_csv('shock_index_labels_mimic_all_events_rolling5min.csv.gz', index=False, compression='gzip')